In [10]:
import numpy as np
import librosa
import matplotlib.pyplot as plt
import soundfile as sf
from scipy.signal import lfilter
from noise import pnoise1

In [11]:
# Load audio
y, sr = librosa.load("voice.wav", sr=None)

In [12]:
# Parameters
frame_length = int(0.03 * sr)   # 30 ms
hop_length = frame_length // 2  # 50% overlap
order = 16

In [13]:
# Pad signal (important for clean framing)
y_padded = np.pad(y, (0, frame_length), mode='constant')

In [14]:
# Frame the signal
frames = librosa.util.frame(y_padded, frame_length=frame_length, hop_length=hop_length)

In [15]:
# Prepare output buffer
output = np.zeros(len(y_padded))
residual_output = np.zeros(len(y_padded))
norm = np.zeros(len(y_padded))

In [16]:
# Window
window = np.hamming(frame_length)

In [ ]:
# Overlap-add reconstruction
for i in range(frames.shape[1]):
    frame = frames[:, i]
    frame = frame * window

    # LPC analysis
    a = librosa.lpc(frame, order=order)

    # Residual (inverse filter)
    residual = lfilter(a, [1.0], frame)
    
    scale = 0.5  # controls how fast the noise evolves

    t = i * scale
    n = pnoise1(t)  # range ≈ [-1, 1]

    # map to [0.2, 0.8]
    alpha_min = 0.01
    alpha_max = 0.99 
    alpha = alpha_min + (n + 1) / 2 * (alpha_max - alpha_min)

    if i > 0:
        residual = (1 - alpha) * residual + alpha * prev_residual

    prev_residual = residual

    # Resynthesis
    reconstructed = lfilter([1.0], a, residual)

    start = i * hop_length

    # accumulate reconstructed
    output[start:start + frame_length] += reconstructed * window

    # accumulate residual
    residual_output[start:start + frame_length] += residual * window

    # normalization tracker
    norm[start:start + frame_length] += window**2

In [ ]:
output /= (norm + 1e-8)
residual_output /= (norm + 1e-8)

sf.write("reconstructed2.wav", output, sr)
sf.write("residual.wav", residual_output, sr)